# Building Spherical Unet `

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import xarray as xr
import healpy as hp
import gc
import sys

## Clear GPU memory

In [21]:
def clear_memory():
    """Clear GPU and CPU memory"""
    vars_to_delete = ['model', 'unet', 'x_tensor', 'output', 'x_multi_band', 'spectral_data', 'neighbor_indices']
    for var in vars_to_delete:
        if var in globals():
            del globals()[var]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("Memory cleared")

In [22]:
# Clear memory before starting
clear_memory()

Memory cleared


In [23]:
# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Load Data (Healpix .Zarr)

In [24]:
ds_healpix = xr.open_dataset("/home/ubuntu/project/sentinel-2-dggs-ai-processor/src/notebook/healpix_20m.zarr", engine='zarr')
available_cell_ids = ds_healpix.cell_ids.values
print(f"Number of available HEALPix cells: {len(available_cell_ids):,}")

Number of available HEALPix cells: 540,155


In [25]:
ds_healpix

<xarray.Dataset> Size: 30MB
Dimensions:    (bands: 6, cells: 540155)
Coordinates:
  * bands      (bands) <U3 72B 'b05' 'b06' 'b07' 'b11' 'b12' 'b8a'
    cell_ids   (cells) int64 4MB 46370881397 46370881398 ... 46729831051
Dimensions without coordinates: cells
Data variables:
    Sentinel2  (bands, cells) float64 26MB ...

In [26]:
# Parameters
level = 18
band_list = ds_healpix.bands.values
in_channels = len(band_list)
out_channels = len(band_list)
stride = 1

In [27]:
print(f"Model config: {in_channels} input channels → {out_channels} output channels")

Model config: 6 input channels → 6 output channels


## NeighborIndexProcessor

This is the core innovation that separates geometric processing from the neural network. The class handles all HEALPix-specific computations outside the model, making it reusable across different data chunks. It takes a HEALPix level (19 in my case) and nest parameter, then provides methods to compute neighbor relationships for any set of cell IDs. The processor includes intelligent caching to avoid recomputing identical neighbor patterns, which is crucial when processing multiple similar chunks. The key insight here is that neighbor relationships depend only on the cell IDs and geometric properties, not on the actual data values, so this can be pre-computed and cached.


The build_neighbor_index function constructs a consistent 9-cell neighbourhood for each HEALPix cell in a dataset at a given resolution level. For each input cell_id, it uses the HEALPix library (Healpy) to retrieve the 8 adjacent neighboring cells using nested indexing. These 8 neighbors, together with the central cell, form a 3×3 patch structure—mimicking the receptive field of a 2D convolutional kernel.

While the HEALPix sphere is globally continuous, a real-world dataset may only cover a limited subset of it. As a result, some of the 8 neighbors returned by Healpy may not be present in the dataset. To ensure consistent patch size across all cells, the function performs an intersection between the neighbor list and the available cells in the dataset. Any missing or invalid neighbors are replaced with the central cell ID, effectively simulating 'same' padding at the borders.

This approach guarantees that each cell is associated with a 9-element patch (center + 8 neighbors), making the data suitable for spatial operations such as spherical convolutions. The output is a 2D NumPy array of shape (N_cells, 9), where each row corresponds to one patch of cell indices ready to be used in downstream convolutional models.


### Information about padding system

In convolutional operations (like in a CNN), we often apply a filter (or "kernel") over a neighborhood of pixels (e.g., a 3×3 grid). But when the filter reaches the edges of the data, it lacks some neighboring values. To avoid losing spatial coverage or changing the output size, padding is applied — typically by adding zeros or repeating values around the edges.

In our HEALPix-based spherical data:

- Each cell has up to 8 neighboring cells.
- cells near the "edges" (e.g., at poles or borders in the pixelization) might not have all 8 valid neighbors.
- HEALPix marks these missing neighbors with -1.

In [28]:
class NeighborIndexProcessor:
    """External processor for building neighbor indices - OUTSIDE the model"""

    def __init__(self, level, nest=True):
        self.level = level
        self.NSIDE = 2 ** level
        self.nest = nest
        self.cache = {}

    def build_neighbor_indices(self, available_cell_ids, stride=1):
        """Build neighbor indices for given cell IDs and stride"""
        # Create cache key
        cache_key = (tuple(sorted(available_cell_ids)), stride)

        if cache_key in self.cache:
            return self.cache[cache_key]

        available_cell_set = set(available_cell_ids)
        neighbor_indices = []

        # Create cell_id to data_index mapping
        cell_to_data_idx = {cell_id: i for i, cell_id in enumerate(available_cell_ids)}

        # Apply stride to center cell list
        center_cells = available_cell_ids[::stride]

        for cell_id in center_cells:
            neighbors = hp.get_all_neighbours(self.NSIDE, cell_id, nest=self.nest)

            # Validate each neighbor; replace invalid or missing with center
            valid_neighbors = [
                n if (n != -1 and n in available_cell_set) else cell_id
                for n in neighbors
            ]

            patch = [cell_id] + valid_neighbors  # Center + 8 neighbors

            # Convert to data indices
            patch_data_indices = [cell_to_data_idx[cell_id] for cell_id in patch]
            neighbor_indices.append(patch_data_indices)

        # Convert to tensor
        data_neighbor_indices = torch.tensor(neighbor_indices, dtype=torch.long)

        # Cache the result
        self.cache[cache_key] = data_neighbor_indices

        return data_neighbor_indices

    def clear_cache(self):
        """Clear the cache"""
        self.cache.clear()

In [29]:
print("Creating neighbor processor...")
neighbor_processor = NeighborIndexProcessor(level=level, nest=True)
neighbor_indices = neighbor_processor.build_neighbor_indices(available_cell_ids, stride)
print(f"Generated {neighbor_indices.shape[0]:,} patches")

Creating neighbor processor...
Generated 540,155 patches


## Verification of kernel generation

In [30]:
sys.path.append('../')
import lonboard
from utils.healpix_plot import *

In [31]:
import xdggs

ds = ds_healpix.pipe(xdggs.decode)

In [32]:
import random

rand_index = random.randint(0, len(neighbor_indices) - 1)
print(rand_index)

92098


In [33]:
# Use of tanh to concentrate the scale variation for the lower values
lonboard.Map(
    [
        exploire_layer(
            ds.Sentinel2.sel(bands=band_list[-1]).compute(),
            alpha=0.10,
            cmap='viridis'
        ),
        exploire_layer(
            ds.Sentinel2.sel(bands=band_list[-1]).compute()[neighbor_indices[random.randint(0, len(neighbor_indices) - 1)]],
            alpha=0.80,
            cmap='plasma'
        ),
        exploire_layer(
            ds.Sentinel2.sel(bands=band_list[-1]).compute()[neighbor_indices[random.randint(0, len(neighbor_indices) - 1)]],
            alpha=0.80,
            cmap='plasma'
        ),
        exploire_layer(
            ds.Sentinel2.sel(bands=band_list[-1]).compute()[neighbor_indices[random.randint(0, len(neighbor_indices) - 1)]],
            alpha=0.80,
            cmap='plasma'
        ),

    ]
)

Map(custom_attribution='', layers=(SolidPolygonLayer(filled=True, get_fill_color=arro3.core.ChunkedArray<Fixed…

## Spherical Conv
### **SphericalConv: The Core Building Block of Spherical CNNs**

The `SphericalConv` class represents is an adapted convolutional neural networks to spherical data structures, specifically designed for HEALPix-organized Earth observation data. Unlike traditional CNNs that operate on regular Euclidean grids with fixed spatial relationships, this class addresses the unique challenge of processing data on a sphere where neighborhoods are determined. The core Class lies in its separation of geometric processing from neural computation: instead of hard-coding specific cell relationships during initialization, the class accepts externally computed `neighbor_indices` that define the 3×3 spherical neighborhoods for each location. During the forward pass, the class performs a batch operation where it simultaneously extracts all spherical patches using tensor indexing (`x[:, :, neighbor_indices]`), creating a tensor of shape `[batch, channels, N_patches, 9]` where each patch contains a center cell and its 8 HEALPix neighbors. These patches are then reshaped into a flattened sequence and processed by a 1D convolution with kernel size 9 and stride 9, effectively applying the same learned spatial filter to every 3×3 spherical neighborhood in parallel. This design enables the network to learn local spherical features while maintaining computational efficiency through GPU parallelization. The class thus serves as the fundamental building block for constructing larger spherical architectures such as U-Nets.

## Integration with Larger Architectures

This `SphericalConv` serves as the fundamental building block for:

- **SphericalConvBlock**: Adds batch normalization and activation functions
- **SphericalDoubleConvBlock**: Implements the standard double convolution pattern
- **SphericalUNet**: Constructs encoder-decoder architectures with skip connections
- **Custom architectures**: Any spherical CNN variant requiring local spatial processing

## Key Advantages

1. **Geometric accuracy**: Respects true spherical relationships rather than projected approximations
2. **Flexibility**: Works with any HEALPix resolution and arbitrary regional coverage
3. **Efficiency**: Leverages modern GPU architectures for parallel processing
4. **Modularity**: Clean separation between geometric preprocessing and neural computation
5. **Compatibility**: Integrates seamlessly with standard PyTorch training pipelines


In [34]:
class SphericalConv(nn.Module):
    """
    Spherical Convolution layer for HEALPix-organized Earth observation data.

    This layer performs convolution operations on spherical data by processing 3×3
    neighborhoods defined by HEALPix geometry. Unlike traditional CNNs that operate
    on regular Euclidean grids, this implementation handles the irregular spatial
    relationships inherent to spherical surfaces through externally computed neighbor
    indices.

    The core innovation is the separation of geometric processing (neighbor finding)
    from neural computation (convolution), enabling flexible processing of arbitrary
    HEALPix regions while maintaining computational efficiency through GPU parallelization.

    Architecture:
    - Uses 1D convolution with kernel_size=9 and stride=9
    - Processes flattened 3×3 spherical patches (center + 8 neighbors = 9 values)
    - Applies the same learned spatial filter to all patches simultaneously
    - Maintains one-to-one correspondence: N input patches → N output features

    Parameters
    ----------
    in_channels : int
        Number of input channels (e.g., spectral bands in satellite imagery).
    out_channels : int
        Number of output feature channels to be learned by the convolution.
    bias : bool, optional
        If True, adds a learnable bias term to the convolution output. Default: True.

    Attributes
    ----------
    conv : nn.Conv1d
        1D convolution layer with kernel_size=9, stride=9 that processes flattened
        spherical patches.

    Notes
    -----
    - Requires externally computed neighbor_indices that define 3×3 spherical
      neighborhoods for each spatial location
    - All patches are processed in parallel for computational efficiency
    - Output preserves spatial correspondence: output[i] corresponds to input patch[i]
    - Handles device compatibility automatically (CPU/GPU synchronization)

    Examples
    --------
    >>> # Initialize spherical convolution layer
    >>> conv_layer = SphericalConv(in_channels=4, out_channels=64)
    >>>
    >>> # Input: [batch_size, channels, n_cells]
    >>> x = torch.randn(1, 4, 10000)  # 1 batch, 4 bands, 10k HEALPix cells
    >>>
    >>> # Neighbor indices: [n_patches, 9] - precomputed 3×3 neighborhoods
    >>> neighbor_indices = torch.randint(0, 10000, (5000, 9))  # 5k patches
    >>>
    >>> # Forward pass
    >>> output = conv_layer(x, neighbor_indices)
    >>> print(output.shape)  # [1, 64, 5000] - 64 features for 5k patches

    Mathematical Operation
    ----------------------
    For each patch i with 9 cells [c₀, c₁, ..., c₈] and learned weights [w₀, w₁, ..., w₈]:

        output[i] = Σⱼ(cⱼ × wⱼ) + bias

    where c₀ is the center cell and c₁-c₈ are the 8 HEALPix neighbors.

    See Also
    --------
    SphericalConvBlock : Spherical convolution with batch normalization and activation
    SphericalDoubleConvBlock : Double spherical convolution block for U-Net architectures
    NeighborIndexProcessor : Utility for computing HEALPix neighbor relationships
    """

    def __init__(self, in_channels, out_channels, bias=True):
        super(SphericalConv, self).__init__()

        # Only the convolution layer - no neighbor processing
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=9, stride=9, bias=bias)

        # Initialize weights
        nn.init.kaiming_normal_(self.conv.weight)
        if bias:
            nn.init.constant_(self.conv.bias, 0.0)

    def forward(self, x, neighbor_indices):
        """
        Perform spherical convolution on input data using precomputed neighbor indices.

        This method extracts 3×3 spherical neighborhoods from the input tensor based on
        the provided neighbor indices, then applies 1D convolution to process all patches
        in parallel. Each patch represents a local spherical neighborhood where the
        spatial relationships are defined by HEALPix geometry.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape [batch_size, in_channels, n_cells] containing
            the data values for all HEALPix cells in the region.
        neighbor_indices : torch.Tensor
            Precomputed neighbor indices of shape [n_patches, 9] where each row
            contains the data indices for one 3×3 spherical neighborhood. The first
            index in each row is the center cell, followed by 8 neighbor indices.

        Returns
        -------
        torch.Tensor
            Output feature tensor of shape [batch_size, out_channels, n_patches]
            where each spatial location corresponds to the convolution result for
            one spherical patch.

        Raises
        ------
        RuntimeError
            If input tensor and neighbor_indices have incompatible devices.
        IndexError
            If neighbor_indices contain values outside the valid range [0, n_cells).

        Notes
        -----
        Processing Pipeline:
        1. Device synchronization: Move neighbor_indices to same device as input
        2. Patch extraction: Extract all 3×3 neighborhoods simultaneously
        3. Tensor reshaping: Flatten patches for 1D convolution processing
        4. Convolution: Apply learned spatial filters to all patches in parallel

        The patch extraction step uses advanced tensor indexing to efficiently
        gather all required neighborhoods in a single operation, enabling
        massive parallelization on GPU hardware.

        Examples
        --------
        >>> conv = SphericalConv(in_channels=3, out_channels=16)
        >>> x = torch.randn(2, 3, 1000)  # 2 batches, 3 channels, 1000 cells
        >>> indices = torch.randint(0, 1000, (500, 9))  # 500 patches
        >>> output = conv(x, indices)  # Shape: [2, 16, 500]
        """
        batch_size, n_channels, n_cells = x.shape

        # Move neighbor_indices to same device as input
        if neighbor_indices.device != x.device:
            neighbor_indices = neighbor_indices.to(x.device)

        # Extract patches using the neighbor indices
        # Shape: [B, C_in, N_patches, 9]
        patches = x[:, :, neighbor_indices]

        # Visualize some example patches
        print(f"Example patches (first 3):")
        for i in range(min(3, patches.shape[2])):
            patch_data = patches[0, 0, i]  # First batch, first channel, patch i
            patch_indices = neighbor_indices[i]
            print(f"Patch {i}: indices {patch_indices[:3].cpu().numpy()}... → values {patch_data[:3].detach().cpu().numpy()}...")

        # Reshape to [B, C_in, N_patches * 9] for Conv1d
        patches_flat = patches.view(batch_size, n_channels, -1)

        # Apply convolution
        output = self.conv(patches_flat)

        return output

In [35]:
Sc = SphericalConv(in_channels=in_channels, out_channels=out_channels,)

In [36]:
# Load and prepare data
spectral_data = []
for band in band_list:
    band_data = ds_healpix.Sentinel2.sel(bands=band).compute().values
    spectral_data.append(band_data)

In [38]:
x_multi_band = np.stack(spectral_data, axis=0)
x_tensor = torch.tensor(x_multi_band, dtype=torch.float32).unsqueeze(0)
print(f"Input tensor shape: {x_tensor.shape}")
print("Running forward pass...")
Sc.eval()
with torch.no_grad():
    output = Sc(x_tensor, neighbor_indices)
    print(f"Output tensor shape: {output.shape}")

output[0,0,0]

Input tensor shape: torch.Size([1, 6, 540155])
Running forward pass...
Example patches (first 3):
Patch 0: indices [0 0 1]... → values [0.1292 0.1292 0.1259]...
Patch 1: indices [1 1 1]... → values [0.1259 0.1259 0.1259]...
Patch 2: indices [2 1 5]... → values [0.11375 0.1259  0.08885]...
Output tensor shape: torch.Size([1, 6, 540155])


tensor(-0.1507)

## Spherical Convolution Block

In [39]:
class SphericalConvBlock(nn.Module):
    """
    Spherical Convolution Block with batch normalization and ReLU activation.

    This block combines spherical convolution with standard deep learning components
    (batch normalization and ReLU activation) to create a robust building block for
    spherical neural networks. It follows the widely-used pattern of Conv → BatchNorm → ReLU
    that has proven effective in modern deep learning architectures.

    Architecture:
    - SphericalConv: Processes 3×3 spherical neighborhoods using HEALPix geometry
    - BatchNorm1d: Normalizes feature distributions for stable training
    - ReLU: Introduces non-linearity for learning complex spatial patterns

    Parameters
    ----------
    in_channels : int
        Number of input channels (e.g., spectral bands in satellite imagery).
    out_channels : int
        Number of output feature channels to be learned by the spherical convolution.

    Attributes
    ----------
    conv : SphericalConv
        Spherical convolution layer that processes 3×3 HEALPix neighborhoods.
    bn : nn.BatchNorm1d
        Batch normalization layer applied along the channel dimension.
    relu : nn.ReLU
        ReLU activation function with in-place operation for memory efficiency.

    Notes
    -----
    - Batch normalization operates on the channel dimension, normalizing across
      all spatial locations (patches) within each batch
    - ReLU activation is applied in-place for memory efficiency
    - The block preserves spatial correspondence: input patch i → output feature i
    - Designed as a drop-in replacement for standard Conv2d blocks in CNNs

    Processing Pipeline
    -------------------
    1. Spherical Convolution: Extract and process 3×3 spherical neighborhoods
    2. Batch Normalization: Normalize feature distributions across spatial locations
    3. ReLU Activation: Apply non-linear activation function

    The batch normalization step is particularly important for spherical networks as
    it helps stabilize training when processing irregular spatial arrangements and
    varying neighborhood sizes that can occur at HEALPix boundaries.

    See Also
    --------
    SphericalConv : Underlying spherical convolution operation
    SphericalDoubleConvBlock : Double convolution block for U-Net architectures
    SphericalUNet : Complete U-Net architecture using spherical convolution blocks
    """

    def __init__(self, in_channels, out_channels):
        super(SphericalConvBlock, self).__init__()

        self.conv = SphericalConv(in_channels, out_channels)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()

    def forward(self, x, neighbor_indices):
        """
        Forward pass through spherical convolution block.

        Processes input through spherical convolution, batch normalization, and
        ReLU activation in sequence. This creates normalized, non-linear feature
        representations that are suitable for building deeper spherical networks.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape [batch_size, in_channels, n_cells] containing
            the data values for all HEALPix cells in the region.
        neighbor_indices : torch.Tensor
            Precomputed neighbor indices of shape [n_patches, 9] defining 3×3
            spherical neighborhoods for convolution processing.

        Returns
        -------
        torch.Tensor
            Output feature tensor of shape [batch_size, out_channels, n_patches]
            containing normalized and activated features for each spherical patch.
            All values are non-negative due to ReLU activation.

        Notes
        -----
        The forward pass applies three sequential operations:

        1. **Spherical Convolution**: Extracts spatial features from 3×3 HEALPix
           neighborhoods using learned convolutional filters

        2. **Batch Normalization**: Normalizes the feature distributions by:
           - Computing mean and variance across spatial dimensions (patches)
           - Applying learnable scale and shift parameters
           - Improving training stability and convergence speed

        3. **ReLU Activation**: Introduces non-linearity by setting negative
           values to zero, enabling the network to learn complex spatial patterns

        The batch normalization is particularly beneficial for spherical data as
        it helps handle the irregular spatial arrangements and boundary effects
        inherent in HEALPix tessellations.
        """
        x = self.conv(x, neighbor_indices)
        x = self.bn(x)
        x = self.relu(x)
        return x

In [40]:
Scb = SphericalConvBlock(in_channels=in_channels, out_channels=out_channels)

In [41]:
Scb.eval()
with torch.no_grad():
    output = Scb(x_tensor, neighbor_indices)
    print(f"Output tensor shape: {output.shape}")

Example patches (first 3):
Patch 0: indices [0 0 1]... → values [0.1292 0.1292 0.1259]...
Patch 1: indices [1 1 1]... → values [0.1259 0.1259 0.1259]...
Patch 2: indices [2 1 5]... → values [0.11375 0.1259  0.08885]...
Output tensor shape: torch.Size([1, 6, 540155])


## Double Convolution Block 

Applies two sequential spherical convolution blocks (Conv→BN→ReLU → Conv→BN→ReLU)
    following the standard U-Net double convolution pattern. This design allows the
    network to learn more complex spatial features within spherical neighborhoods.

In [42]:
class SphericalDoubleConvBlock(nn.Module):
    """
    Double spherical convolution block for deeper feature extraction.

    Applies two sequential spherical convolution blocks (Conv→BN→ReLU → Conv→BN→ReLU)
    following the standard U-Net double convolution pattern. This design allows the
    network to learn more complex spatial features within spherical neighborhoods.

    Parameters
    ----------
    in_channels : int
        Number of input channels.
    out_channels : int
        Number of output channels for both convolution blocks.

    Notes
    -----
    Architecture: Input → SphericalConvBlock → SphericalConvBlock → Output
    - First block: in_channels → out_channels
    - Second block: out_channels → out_channels (refinement)
    """

    def __init__(self, in_channels, out_channels):
        super(SphericalDoubleConvBlock, self).__init__()

        self.conv1 = SphericalConvBlock(in_channels, out_channels)
        self.conv2 = SphericalConvBlock(out_channels, out_channels)

    def forward(self, x, neighbor_indices):
        """
        Apply double spherical convolution.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor [batch_size, in_channels, n_cells].
        neighbor_indices : torch.Tensor
            Precomputed neighbor indices [n_patches, 9].

        Returns
        -------
        torch.Tensor
            Output features [batch_size, out_channels, n_patches].
        """
        x = self.conv1(x, neighbor_indices)
        x = self.conv2(x, neighbor_indices)
        return x

## Spherical Transpose Convolution

Spherical Transpose Convolution provides learnable upsampling for decoder paths in spherical U-Net like architectures, serving as the counterpart to the forward SphericalConv operation. While forward spherical convolution extracts features from 3×3 HEALPix neighborhoods and reduces spatial resolution, transpose convolution performs the inverse operation by learning to reconstruct higher-resolution feature maps from compressed representations. The implementation uses 1D transpose convolution with the same kernel size (9) and stride (9) configuration as the forward pass. Unlike forward convolution which requires neighbor indices to define spherical relationships, transpose convolution operates purely in feature space, making the neighbor_indices parameter unused but maintained for API consistency across all spherical layers and potentially used for future development (compression)

In [43]:
class SphericalConvTranspose(nn.Module):
    """
    Core spherical transpose convolution for learnable upsampling.

    Performs learned upsampling using 1D transpose convolution with the same
    kernel and stride configuration as SphericalConv (kernel_size=9, stride=9).
    This is the core operation without normalization or activation functions.

    Parameters
    ----------
    in_channels : int
        Number of input channels.
    out_channels : int
        Number of output channels after upsampling.
    bias : bool, optional
        Whether to include learnable bias term. Default: True.

    Notes
    -----
    - Core transpose convolution operation only
    - Uses same kernel_size=9 and stride=9 as forward spherical convolution
    - Applies Kaiming initialization for stable training
    - neighbor_indices parameter unused but maintained for API consistency
    """

    def __init__(self, in_channels, out_channels, bias=True):
        super(SphericalConvTranspose, self).__init__()

        # ConvTranspose1d for upsampling
        self.conv_transpose = nn.ConvTranspose1d(
            in_channels, out_channels,
            kernel_size=9, stride=9, bias=bias
        )

        # Initialize weights
        nn.init.kaiming_normal_(self.conv_transpose.weight)
        if bias:
            nn.init.constant_(self.conv_transpose.bias, 0.0)

    def forward(self, x, neighbor_indices=None):
        """
        Apply transpose convolution for upsampling.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor [batch_size, in_channels, n_patches].
        neighbor_indices : torch.Tensor, optional
            Unused parameter maintained for API consistency.

        Returns
        -------
        torch.Tensor
            Upsampled output [batch_size, out_channels, n_patches_upsampled].
        """
        return self.conv_transpose(x)

In [44]:
class SphericalConvTransposeBlock(nn.Module):
    """
    Spherical transpose convolution block with batch normalization and ReLU.

    Combines spherical transpose convolution, batch normalization, and ReLU activation
    into a single reusable component following the standard pattern:
    ConvTranspose → BatchNorm → ReLU.

    Parameters
    ----------
    in_channels : int
        Number of input channels.
    out_channels : int
        Number of output channels after upsampling.
    bias : bool, optional
        Whether to include learnable bias term. Default: True.

    Notes
    -----
    Architecture: Input → SphericalConvTranspose → BatchNorm1d → ReLU → Output
    """

    def __init__(self, in_channels, out_channels, bias=True):
        super(SphericalConvTransposeBlock, self).__init__()

        self.conv_transpose = SphericalConvTranspose(in_channels, out_channels, bias)
        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, neighbor_indices=None):
        """
        Apply transpose convolution block for upsampling.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor [batch_size, in_channels, n_patches].
        neighbor_indices : torch.Tensor, optional
            Unused parameter maintained for API consistency.

        Returns
        -------
        torch.Tensor
            Upsampled output [batch_size, out_channels, n_patches_upsampled].
        """
        x = self.conv_transpose(x, neighbor_indices)
        x = self.bn(x)
        x = self.relu(x)
        return x